<a href="https://colab.research.google.com/github/minarmaing/Handspell-v02/blob/main/kazim_mediapipe_xgboost_colab_v03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installing Python Libraries
To install a library, use the `!` prefix followed by `pip install <library_name>`.

For example, to install the `requests` library (or any other package from PyPI):

In [ ]:
%pip install mediapipe xgboost scikit-learn pillow numpy matplotlib tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Download the zip dataset directly into Colab's local environment
!wget -O dataset.zip "https://data.mendeley.com/public-files/datasets/j4y5w2c8w9/files/28201373-c176-4b20-8f5e-c963d3ca1564/file_downloaded"

# 2. Unzip the file silently into a target folder named 'dataset'
!unzip -q dataset.zip -d dataset/

# 3. Clean up the zip file to save workspace storage
!rm dataset.zip

print("Dataset successfully downloaded and unzipped!")


In [ ]:
import os
print(os.listdir('dataset'))

In [ ]:
# 1. Install the unrar extraction utility
!apt-get install unrar -y

# 2. Download the RAR dataset directly into Colab
!wget -O dataset.rar "https://data.mendeley.com/public-files/datasets/xs6mvhx6rh/files/8c504c17-1765-4935-9f4f-4a2aa68cb9f0/file_downloaded"

# 3. Extract the RAR file silently into a target folder named 'dataset_rar'
# Note: 'x' extracts with full paths, '-q' runs it quietly
!unrar x dataset.rar dataset_rar/

# 4. Clean up the compressed archive to free up space
!rm dataset.rar

print("RAR Dataset successfully downloaded and extracted!")


In [ ]:
import os
print(os.listdir('dataset_rar'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import Path from pathlib to handle filesystem paths platform-independently
from pathlib import Path
# Import urlretrieve to download the MediaPipe hand-landmarker model file from a URL
from urllib.request import urlretrieve

# Import NumPy for high-performance numerical operations and array manipulation
import numpy as np
# Import MediaPipe for hand tracking and 3D landmarker extraction
import mediapipe as mp
# Import Matplotlib's pyplot for rendering training preview plots
import matplotlib.pyplot as plt

# Import Image for loading images, and ImageOps for auto-correcting orientation tags
from PIL import Image, ImageOps
# Import tqdm to display real-time progress bars during long extraction loops
from tqdm.auto import tqdm
# Import train_test_split to divide the dataset into stratified train and validation splits
from sklearn.model_selection import train_test_split
# Import classification metrics to compute model accuracy and per-class reports
from sklearn.metrics import accuracy_score, classification_report
# Import XGBClassifier to train our gradient-boosted decision tree classification model
from xgboost import XGBClassifier

# Define the local folder containing the primary training ASL alphabet dataset
TRAIN_ROOT = Path("dataset/asl_dataset")
# Define the local folder containing the external ASLYset validation dataset
EXTERNAL_ROOT = Path("dataset_rar/ASLYset/images")

# Define the 24 letters of the ASL alphabet modeled (excluding J and Z due to dynamic motion)
LETTERS = "ABCDEFGHIKLMNOPQRSTUVWXY"
# Build a dictionary mapping each uppercase letter to a class index integer (0 to 23)
LABEL_MAP = {letter: index for index, letter in enumerate(LETTERS)}

# Configure the local storage directory and path for the MediaPipe model asset file
HAND_MODEL = Path("models/hand_landmarker.task")
# Generate the parent directory structure if it does not already exist on disk
HAND_MODEL.parent.mkdir(parents=True, exist_ok=True)

# Source URL pointing to the static, optimized float16 MediaPipe hand landmarker model bundle
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/"
    "hand_landmarker.task"
)

# If the model file is not already present locally, download it directly from the Google API storage
if not HAND_MODEL.exists():
    urlretrieve(MODEL_URL, HAND_MODEL)

# Confirm that the MediaPipe hand landmarker model is successfully ready and cached
print("MediaPipe model:", HAND_MODEL)

In [ ]:
from pathlib import Path
import shutil

# Define the target backup path on your mounted Google Drive
drive_models = Path("/content/drive/MyDrive/HandSpell/models")
# Recursively create the directories on Google Drive if they don't exist yet
drive_models.mkdir(parents=True, exist_ok=True)

# Define the final destination file path inside the Google Drive models directory
destination = drive_models / "hand_landmarker.task"

# Copy the downloaded model task bundle from the local environment to Google Drive, preserving file metadata
shutil.copy2(HAND_MODEL, destination)

# Print confirmation status and the copied file's size in bytes for integrity checking
print("Saved to:", destination)
print("File size:", destination.stat().st_size, "bytes")

In [ ]:
from google.colab import auth
auth.authenticate_user()

BUCKET_NAME = "handspell-asl-models-bucket"

# Run this immediately after training finishes
!gsutil -m cp -r models/ gs://{BUCKET_NAME}/
print("Securely uploaded to GCP before disconnect!")

In [ ]:
def find_images(root):
    # Convert the input path string into a robust pathlib.Path object
    root = Path(root)

    # Raise an error immediately if the specified target directory does not exist
    if not root.is_dir():
        raise FileNotFoundError(root)

    # Initialize empty lists to store file paths and their corresponding labels
    paths, labels = [], []

    # Recursively traverse all files and subdirectories within the root directory
    for path in sorted(root.rglob("*")):
        # Skip any directories; we are only interested in files
        if not path.is_file():
            continue

        # Filter files to only keep images with .jpg, .jpeg, or .png extensions
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        # Extract the name of the parent folder (the letter folder, e.g., 'A') and capitalize it
        letter = path.parent.name.upper()

        # If the detected letter is a valid category in our predefined LABEL_MAP
        if letter in LABEL_MAP:
            # Record the full file path
            paths.append(path)
            # Map the letter to its corresponding index integer and add to labels list
            labels.append(LABEL_MAP[letter])

    # If no valid images were found in the folder structure, raise a descriptive value error
    if not paths:
        raise ValueError(f"No images in supported letter folders: {root}")

    # Return the list of image paths and a NumPy array of integer labels (as 64-bit integers)
    return paths, np.array(labels, dtype=np.int64)

In [ ]:
def read_rgb(path):
    # Open the image file using Pillow's Image class
    with Image.open(path) as original:
        # Correct orientation based on EXIF camera tags and convert image format to RGBA
        oriented = ImageOps.exif_transpose(original).convert("RGBA")

        # Create a solid white background image matching the dimension of the oriented image
        background = Image.new("RGBA", oriented.size, (255, 255, 255, 255))

        # Composites oriented image on top of white background to convert transparency smoothly to white
        image = Image.alpha_composite(background, oriented).convert("RGB")

        # Resize the image dynamically to fit within a 1024x1024 box, preserving aspect ratio for faster extraction
        image.thumbnail((1024, 1024), Image.Resampling.LANCZOS)

        # Convert the processed PIL Image object into a standard NumPy array for MediaPipe compatibility
        return np.array(image)

In [ ]:
def create_landmarker():
    # Set up configuration options for the MediaPipe HandLandmarker API
    options = mp.tasks.vision.HandLandmarkerOptions(
        # Provide the local path to the pretrained MediaPipe hand detection model file (.task)
        base_options=mp.tasks.BaseOptions(
            model_asset_path=str(HAND_MODEL)
        ),
        # Configure running mode to accept static images instead of real-time video stream
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
        # Set maximum hands to detect in any single image to 1
        num_hands=1,
        # Establish threshold of 50% minimum confidence for hand detection
        min_hand_detection_confidence=0.5,
        # Establish threshold of 50% minimum confidence to assert presence of hand keypoints
        min_hand_presence_confidence=0.5,
    )

    # Return a configured and running instance of the MediaPipe HandLandmarker client
    return mp.tasks.vision.HandLandmarker.create_from_options(options)

In [ ]:
def get_features(rgb, landmarker):
    # Pack the NumPy RGB array into a MediaPipe Image container, ensuring contiguous memory layout
    image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=np.ascontiguousarray(rgb),
    )

    # Run the feature detector to locate the hand skeleton in the image
    result = landmarker.detect(image)

    # If MediaPipe fails to detect a hand structure, exit gracefully and return None
    if not result.hand_landmarks:
        return None, None

    # Extract relative normalized coordinates (x, y, z) for each of the 21 key points
    points = np.array(
        [[point.x, point.y, point.z] for point in result.hand_landmarks[0]],
        dtype=np.float32,
    )

    # Get original image pixel bounds (height, width, channels)
    height, width = rgb.shape[:2]

    # De-normalize coordinates back to original image absolute pixel space scale
    points *= np.array([width, height, width], dtype=np.float32)

    # Subtract wrist coordinate (index 0) from all points to make wrist the translation origin (0, 0, 0)
    relative = points - points[0]

    # Compute Euclidean distance between wrist (0) and middle knuckle (9) to serve as a scale metric
    scale = np.linalg.norm(relative[9])

    # Prevent division by zero errors for invalid hand scales
    if scale < 1e-6:
        return None, None

    # Rescale relative hand metrics and flatten the 21 x 3 coordinate array into a 1D vector of 63 values
    features = (relative / scale).reshape(-1)

    # Return both the flattened 63 features and raw MediaPipe keypoints for downstream tasks
    return features, result.hand_landmarks[0]

In [ ]:
def extract_features(paths):
    # Initialize empty 2D array of NaNs matching dataset size (each row receives a 63-dim feature vector)
    features = np.full((len(paths), 63), np.nan, dtype=np.float32)
    # List to record paths of images where landmark extraction failed
    failures = []

    # Use Python context manager to initialize the landmarker once for the entire directory loop
    with create_landmarker() as landmarker:
        for index, path in enumerate(tqdm(paths)):
            try:
                # Load the RGB image array safely
                rgb = read_rgb(path)
                # Compute the landmark vector
                vector, _ = get_features(rgb, landmarker)

                # If no hand was found, log a detection failure
                if vector is None:
                    failures.append((str(path), "No usable hand landmarks"))
                else:
                    # Write the successful 63 features into our pre-allocated array
                    features[index] = vector

            # Capture standard filesystem/OS errors, parsing issues, or processing timeouts
            except (OSError, ValueError, RuntimeError) as error:
                failures.append((str(path), str(error)))

    # Return features matrix and tracking list of processing failures
    return features, failures

In [ ]:
# Scan the training directory to retrieve all image file paths and their mapped integer class labels
train_paths, train_labels = find_images(TRAIN_ROOT)

# Instantiate a random number generator with a fixed seed for reproducible image sampling
sample_indices = np.random.default_rng(42).choice(
    len(train_paths),
    size=min(6, len(train_paths)),
    replace=False,
)

# Set up a 2x3 subplot grid canvas using Matplotlib for visualizing 6 random image samples
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Loop over and clean each subplot by turning off administrative axis lines and ticks
for ax in axes.flat:
    ax.axis("off")

# Open the MediaPipe HandLandmarker API using the standard context manager wrapper
with create_landmarker() as landmarker:
    # Iterate concurrently over each coordinate axis and corresponding random image index
    for ax, index in zip(axes.flat, sample_indices):
        # Load the designated file as a standard RGB pixel array
        rgb = read_rgb(train_paths[index])
        # Extract raw landmarks directly using our custom hand-skeleton detector
        _, landmarks = get_features(rgb, landmarker)

        # Display the base RGB image onto the respective grid axis
        ax.imshow(rgb)

        # If the MediaPipe model successfully mapped joint locations within the image bounds
        if landmarks:
            # Fetch actual pixel dimensions to upscale relative landmark outputs
            height, width = rgb.shape[:2]
            # Project and overlay the 21 joints onto the image using bright lime-green dot markers
            ax.scatter(
                [point.x * width for point in landmarks],
                [point.y * height for point in landmarks],
                s=12,
                c="lime",
            )
            # Set status description flag to 'detected'
            status = "detected"
        else:
            # Otherwise, flag that no skeleton landmarks were successfully extracted
            status = "NO HAND"

        # Annotate each plot with the ground-truth target letter and extraction status outcome
        ax.set_title(f"{LETTERS[train_labels[index]]}: {status}")

# Adjust subplot spacing automatically to prevent letter overlays or title cropping
plt.tight_layout()
# Render the completed diagnostic canvas to the notebook output screen
plt.show()

In [ ]:
# Partition the image dataset indices into training (80%) and validation (20%) subsets
# Setting stratify=train_labels guarantees that both splits contain matching proportions of each ASL letter class
train_indices, val_indices = train_test_split(
    np.arange(len(train_paths)),
    test_size=0.2,
    random_state=42,
    stratify=train_labels,
)

# Run the bulk coordinate extraction pipeline to extract 63 relative features from all training images
features_all, training_failures = extract_features(train_paths)

# Isolate the extracted 63-dimensional coordinate feature vectors for the designated training indices
X_train = features_all[train_indices]
# Isolate the corresponding ground-truth class labels (0-23) for the training split
y_train = train_labels[train_indices]

# Isolate the extracted 63-dimensional coordinate feature vectors for the validation indices
X_val = features_all[val_indices]
# Isolate the corresponding ground-truth class labels (0-23) for the validation split
y_val = train_labels[val_indices]

# Create a boolean mask indicating rows where all extracted coordinate values are finite numbers (not NaN)
# MediaPipe failures produce rows of NaNs, which we must filter out prior to fitting the classifier
train_ok = np.isfinite(X_train).all(axis=1)

# Calculate and display the percentage of training images where hand landmark detection succeeded
print(f"Training hand detection: {train_ok.mean():.1%}")
# Output the first three detected pipeline or filesystem failures to assist with diagnostics
print("First loading/detection failures:", training_failures[:3])

# Verify that all 24 required target alphabet classes contain at least one successfully extracted hand
missing = set(range(24)) - set(y_train[train_ok])

# If any class index has zero valid hands detected, raise an error to prevent incomplete model training
if missing:
    raise ValueError(
        f"No usable training images for: {[LETTERS[i] for i in sorted(missing)]}"
    )

In [ ]:
# Initialize the XGBoost classifier with custom hyperparameters optimized for coordinate-based landmark classification
classifier = XGBClassifier(
    # Configure the learning target to output soft class membership probabilities for multiclass datasets
    objective="multi:softprob",
    # Specify the exact number of unique target alphabet letters (classes 0 through 23)
    num_class=24,
    # Define the total number of consecutive decision trees (boosting rounds) to construct
    n_estimators=250,
    # Limit individual tree depth to 4 splits to regularize the model and prevent overfitting
    max_depth=4,
    # Set learning rate (shrinkage value) to scale the step size and slow down convergence for better optimization
    learning_rate=0.05,
    # Randomly subsample 80% of the training instances for each tree to introduce robustness against noise
    subsample=0.8,
    # Randomly select 80% of the 63 coordinate features per tree to prevent single feature dominance
    colsample_bytree=0.8,
    # Activate the fast histogram-based split-finding algorithm to speed up tree growth and reduce RAM consumption
    tree_method="hist",
    # Use multi-class logarithmic loss (cross-entropy) as our validation evaluation metric
    eval_metric="mlogloss",
    # Seed the random number generator to guarantee deterministic, reproducible runs
    random_state=42,
    # Utilize up to 4 parallel processing CPU cores for boosting operations
    n_jobs=4,
)

# Train (fit) the configured XGBoost classifier on our landmark coordinates
# This utilizes boolean masking ('train_ok') to exclude any invalid rows where hand detection failed
classifier.fit(
    X_train[train_ok],
    y_train[train_ok],
)

In [ ]:
def evaluate_pipeline(classifier, features, labels, name):
    # Identify rows where all 63 coordinates are valid, finite numbers (not NaN)
    usable = np.isfinite(features).all(axis=1)

    # Pre-populate all predictions with -1; representing a failure to detect a hand
    # These will automatically count as incorrect predictions since classes are 0 to 23
    predictions = np.full(len(labels), -1, dtype=np.int64)

    # If there is at least one image with a valid detected hand
    if usable.any():
        # Use the trained classifier to predict classes for detected hand features only
        predictions[usable] = classifier.predict(
            features[usable]
        ).astype(np.int64)

    # Print evaluation metadata and dataset diagnostics
    print(f"\n{name}")
    print(f"Images: {len(labels)}")
    print(f"Usable hand detection: {usable.mean():.2%}")
    print(f"Overall accuracy: {(predictions == labels).mean():.2%}")

    # If some hands were detected, calculate classification accuracy strictly on those samples
    if usable.any():
        accuracy = accuracy_score(
            labels[usable], predictions[usable]
        )
        print(f"Accuracy on detected hands: {accuracy:.2%}")

    # Return the final predictions array (containing valid classes or -1 for failures)
    return predictions


# Run the evaluation pipeline on the validation split
val_predictions = evaluate_pipeline(
    classifier, X_val, y_val, "Validation"
)

In [ ]:
# Retrieve all valid image file paths and their matching integer labels from the ASLYset external directory
external_paths, y_external = find_images(EXTERNAL_ROOT)

# Perform bulk feature extraction on all collected external test images to acquire the 63-dimensional relative coordinate vectors
X_external, external_failures = extract_features(external_paths)

# Pass the trained classifier, extracted features, and labels to the evaluation function
# This prints the overall accuracy metrics on the external out-of-domain dataset
external_predictions = evaluate_pipeline(
    classifier,
    X_external,
    y_external,
    "ASLYset external evaluation",
)

In [ ]:
import json
from pathlib import Path

# Define the local output directory for model artifacts
output_dir = Path("models")
# Create the directory structure if it does not already exist
output_dir.mkdir(parents=True, exist_ok=True)

# Export the trained XGBoost classifier structure to a portable JSON file
classifier.save_model(
    str(output_dir / "handspell_xgboost_v1.json")
)

# Build a metadata dictionary documenting the labels, features, and model properties
metadata = {
    "letters": LETTERS,
    "feature_count": 63,
    "normalization": "wrist origin; wrist-to-middle-MCP distance",
    "external_accuracy": 0.6456,
}

# Save the metadata dictionary to a JSON file with pretty formatting
(output_dir / "handspell_xgboost_v1_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Model and metadata saved.")

In [ ]:
import json
from pathlib import Path
import shutil

# Configure folder path to save artifacts on mounted Google Drive
output_dir = Path("/content/drive/MyDrive/Handspell_Models")
# Generate directory structure dynamically if it is not present
output_dir.mkdir(parents=True, exist_ok=True)

# Save the final trained XGBoost model structure as a portable JSON object
classifier.save_model(str(output_dir / "handspell_xgboost_v1.json"))

# Construct dictionary of metadata to document exact training metrics and configuration rules
metadata = {
    "letters": "ABCDEFGHIKLMNOPQRSTUVWXY",
    "feature_count": 63,
    "normalization": "wrist origin; wrist-to-middle-MCP distance",
}

# Write metadata file out to target Google Drive directory with clean JSON indents
(output_dir / "handspell_xgboost_v1_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

# Identify location of locally downloaded MediaPipe model resource
source = Path(HAND_MODEL)
# Set output backup file path on Google Drive
destination = output_dir / "hand_landmarker.task"

# If target drive location is different from local cache, copy the binary file
if source.resolve() != destination.resolve():
    shutil.copy2(source, destination)

# Print verified backup files and physical file sizes written to Google Drive
for path in output_dir.iterdir():
    print(path.name, path.stat().st_size, "bytes")

In [ ]:
from google.colab import auth
auth.authenticate_user()

BUCKET_NAME = "handspell-asl-models-bucket"

# Run this immediately after training finishes
!gsutil -m cp -r models/ gs://{BUCKET_NAME}/
print("Securely uploaded to GCP before disconnect!")

In [ ]:
# Generate and print a comprehensive text-based classification report comparing true labels with external test predictions
print(classification_report(
    # Provide the ground-truth integer labels for the external test set
    y_external,
    # Provide the corresponding class predictions outputted by the pipeline
    external_predictions,
    # Specify the list of valid class indices (0 to 23) to compute metrics for
    labels=np.arange(24),
    # Map target indices back to their original uppercase ASL letters for readability
    target_names=list(LETTERS),
    # Format output values to 3 decimal places for finer evaluation precision
    digits=3,
    # Return 0.0 for division-by-zero occurrences to handle categories with zero predictions cleanly
    zero_division=0,
))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Predefined alphabet string mapping of the 24 modeled ASL uppercase letters
LETTERS = "ABCDEFGHIKLMNOPQRSTUVWXY"

# Map any detection failures (represented as -1) to class index 24 for the confusion matrix display
predictions = np.where(
    external_predictions == -1,
    24,
    external_predictions,
)

# Construct the full list of labels for the plot, appending 'No hand' as the final 25th label
display_labels = list(LETTERS) + ["No hand"]

# Calculate the confusion matrix across all 25 possible outcomes (0-23 for letters, 24 for No hand)
matrix = confusion_matrix(
    y_external,
    predictions,
    labels=np.arange(25),
)

# Initialize a large figure size for high-resolution readability and clear cell annotation
plt.figure(figsize=(16, 13))

# Render the raw counts using a seaborn heatmap with integer formatting, a blue colormap, and labels
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=display_labels,
    yticklabels=display_labels[:24],  # True labels only go up to index 23 (24 classes) since ground truths never contain 'No hand'
    cbar=True,
    square=True
)

# Style the graph title and axes labels for clean final reporting
plt.title("MediaPipe + XGBoost — ASLYset External Test Misclassifications Heatmap", fontsize=16, pad=20)
plt.xlabel("Predicted Label", fontsize=12, labelpad=15)
plt.ylabel("True Label", fontsize=12, labelpad=15)
plt.xticks(rotation=90)
plt.yticks(rotation=0)

# Automatically fit elements comfortably within image boundaries and render the final plot
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import auth
auth.authenticate_user()

# Set your GCP project ID and a unique bucket name
PROJECT_ID = "kazim-wagon-bootcamp"
BUCKET_NAME = "handspell-asl-models-bucket"

# Create the bucket (only needed the first time)
!gcloud config set project {PROJECT_ID}
!gsutil mb -l us-central1 gs://{BUCKET_NAME}/

!gcloud config set project {PROJECT_ID}
# 1. Create a local models folder structure if it doesn't exist
import os
os.makedirs('models', exist_ok=True)

# 2. Upload your entire local models directory into your GCS bucket root
# (Note the exclamation mark '!' and curly braces '{BUCKET_NAME}' required by Colab)
!gsutil -m cp -r models/ gs://{BUCKET_NAME}/


print("All models and metadata successfully synced to Google Cloud Storage!")